In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
# Load JSON from file
with open('data/items.json', 'r') as file:
    dct = json.load(file)
    items = dct["references"]["RefIds"]

with open('data/effects.json', 'r') as file:
    dct = json.load(file)
    effects = dct["references"]["RefIds"]

with open('data/item_types.json', 'r') as file:
    dct = json.load(file)
    item_types = dct["references"]["RefIds"]

with open('data/languages/es.json', 'r') as file:
    names = json.load(file)

with open('data/item_sets.json', 'r') as file:
    dct = json.load(file)
    item_sets = dct["references"]["RefIds"]
    
def get_mapper(dct,context=None,field="rid"):
    mapper = {}
    for d in dct:
        if context is not None:
            mapper[d[context][field]] = d
            continue
        mapper[str(d[field])] = d
    return mapper

mapper_rid = {"items": get_mapper(items),
               "effects": get_mapper(effects),
               "item_types": get_mapper(item_types),
               "item_sets": get_mapper(item_sets),
               "names": names}


In [3]:
# Data parser function
# This function extracts relevant data from the mapper and formats it into a dictionary.
def get_data(mapper,name_field="nameId"):
    data = {}
    for rid, item in mapper.items():
        nameId = str(item["data"].get(name_field, None))
        name = names["entries"].get(nameId, None)
        data[rid] = {"type": item["type"]["class"],
                     "name": name}
        data[rid].update(item["data"].copy())
        # data[rid].update({k:v for k,v in item.items() if k not in data[rid]})
        if "possibleEffects" in data[rid]:
            data[rid]["possibleEffects"] = ",".join([str(i["rid"]) for i in data[rid]["possibleEffects"]["Array"]])
        for field in ["id", "nameId", "typeId", "setId","effectId","characteristic"]:
            if field not in data[rid]:
                continue
            # Ensure that the fields are string
            data[rid][field] = str(data[rid][field])
    return data

In [4]:
def ensure_string_fields(df):
    """Ensure that specific fields in the DataFrame are of string type."""
    for field in ["id", "nameId", "typeId", "set"]:
        if field not in df.columns:
            continue
        # Ensure that the fields are string
        df[field] = df[field].astype(str)

In [5]:
# Effect data
effect_data = get_data(mapper_rid["effects"], name_field="descriptionId")
effect_df = pd.DataFrame.from_dict(effect_data, orient="index")
# Move description to the first column
effect_df = effect_df[["name"] + [col for col in effect_df.columns if col != "name"]]
ensure_string_fields(effect_df)
effect_df.to_csv("data/datasets/effects.csv", index=True)
effect_name_dictionary = effect_df.set_index("id")["name"].to_dict()

In [6]:
# Item type data
item_types = get_data(mapper_rid["item_types"], name_field="nameId")
item_types_df = pd.DataFrame.from_dict(item_types, orient="index")
# Move name to the first column
item_types_df = item_types_df[["name"] + [col for col in item_types_df.columns if col != "name"]]
ensure_string_fields(item_types_df)
item_types_df.to_csv("data/datasets/item_types.csv", index=True)

In [7]:
# Item set data
item_sets = get_data(mapper_rid["item_sets"], name_field="nameId")
item_sets_df = pd.DataFrame.from_dict(item_sets, orient="index")
# Move name to the first column
item_sets_df = item_sets_df[["name"] + [col for col in item_sets_df.columns if col != "name"]]
item_sets_df["items"] = item_sets_df["items"].apply(lambda x: ",".join([str(i) for i in x["Array"]]) if isinstance(x, dict) and "Array" in x else "")
ensure_string_fields(item_types_df)
item_sets_df.to_csv("data/datasets/item_sets.csv", index=True)

In [8]:
# Get a Item to Set mapper
item_to_set_mapper = {}
for rid,row in item_sets_df.iterrows():
    items = row["items"].split(",")
    for item in items:
        if item == "":
            continue
        item_to_set_mapper[item] = rid

In [9]:
# Item data
item_data = get_data(mapper_rid["items"], name_field="nameId")
item_df = pd.DataFrame.from_dict(item_data, orient="index")
# Move name to the first column
item_df = item_df[["name"] + [col for col in item_df.columns if col != "name"]]

# Only change name where there is an effectId
item_df["name"] = item_df["name"].where(item_df["effectId"].isna(),
                                         item_df["effectId"].apply(lambda x: effect_name_dictionary.get(x, "")))
item_df["class"] = item_df["typeId"].apply(lambda x: item_types_df.set_index("id").loc[x, "name"] if item_types_df["id"].isin([x]).any() else x)
item_df["set"] = item_df["id"].apply(lambda x: item_to_set_mapper.get(x, ""))
ensure_string_fields(item_df)
item_df.to_csv("data/datasets/items.csv", index=True)

In [10]:
# Set effects data
set_effects = {}
for rid,row in item_sets_df.iterrows():
    set_effects[rid] = {}
    dct = row["effects"]
    if pd.isna(dct):
        continue
    for idx,v in enumerate(dct["Array"]):
        lst = v["values"]["Array"]
        set_effects[rid][idx+1] = ",".join([str(i["rid"]) for i in lst])

pd.DataFrame.from_dict(set_effects, orient="index").to_csv("data/datasets/set_effects.csv", index=True)


In [11]:
assert False

AssertionError: 

### Interpreting

### Effects 
Characteristic -> effect(res/add/subtract) and element

characteristicOperator -> +/-

isInPercent -> % or not

### Items 

diceNum -> minimum stat

diceSide -> maximum stat

typeId -> type of item (cape, hat, etc)

level -> level

criteria -> requirements

criticalHitBonus -> Bonus with crits

criticalHitProbability -> 0 - 100 %

range

castInLine -> 0/1

apCost -> AP to use

maxCastPerTurn

In [ ]:
item_df[[(item_df["typeId"]==2)|(item_df["typeId"]==3)]]

type
EffectInstanceDice    40068
Items                 19946
Weapons                 747
Name: count, dtype: int64

In [12]:
item_df

,name,type,m_flags,id,nameId,typeId,descriptionId,iconId,level,realWeight,...,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero,class,set
2861093746001846693,Espada de maderucha,Weapons,24712,44,995824,6,995825.0,6007.0,7.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,
2861093746001846694,Espada de hierrucho,Weapons,24712,49,995826,6,33282.0,6002.0,15.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,
2861093746001846695,Espada chifladora,Weapons,24712,55,995827,6,995830.0,6001.0,26.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,
2861093746001846696,La Triste Hoja,Weapons,24712,58,47565,6,47566.0,6005.0,21.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,
2861093746001846697,Espada Dha,Weapons,24712,62,995829,6,1064630.0,6006.0,17.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2861093746004781817,¿Qué hay ahí dentro?,EffectInstanceDice,15,nan,nan,nan,NaN,NaN,NaN,NaN,...,-1.0,-1.0,0.0,"{'cellIds': {'Array': []}, 'shape': 67, 'param...",439.0,1.0,0.0,0.0,NaN,
2861093746004781818,Intercambiable: #1,EffectInstanceDice,15,nan,nan,nan,NaN,NaN,NaN,NaN,...,-1.0,-1.0,0.0,"{'cellIds': {'Array': []}, 'shape': 67, 'param...",0.0,0.0,0.0,0.0,NaN,
2861093746004781819,¿Qué hay ahí dentro?,EffectInstanceDice,15,nan,nan,nan,NaN,NaN,NaN,NaN,...,-1.0,-1.0,0.0,"{'cellIds': {'Array': []}, 'shape': 67, 'param...",440.0,1.0,0.0,0.0,NaN,
2861093746004781820,Intercambiable: #1,EffectInstanceDice,15,nan,nan,nan,NaN,NaN,NaN,NaN,...,-1.0,-1.0,0.0,"{'cellIds': {'Array': []}, 'shape': 67, 'param...",0.0,0.0,0.0,0.0,NaN,


In [ ]:
effect_df[effect_df["name"].str.contains("resistencia.*agua", na=False)]

,name,type,id,descriptionId,iconId,characteristic,category,characteristicOperator,showInTooltip,useDice,...,bonusType,useInFight,effectPriority,effectPowerRate,elementId,isInPercent,hideValueInTooltip,textIconReferenceId,effectTriggerDuration,actionFiltersId
2861093746001578911,#1{{~1~2 a }}#2% resistencia al agua,Effects,211,1067166,0,35,1,+,0,1,...,1,0,9700,6.0,-1,1,0,34,0,"{'Array': [6, 10, 13]}"
2861093746001578916,-#1{{~1~2 a -}}#2% resistencia al agua,Effects,216,1067168,0,35,1,-,0,1,...,-1,0,10200,-3.0,-1,1,0,34,0,"{'Array': [6, 12]}"
2861093746001578931,#1{{~1~2 a }}#2 de resistencia{{~p}}{{~z}} al ...,Effects,241,1067167,0,56,1,+,0,1,...,1,0,10700,2.0,-1,0,0,34,0,"{'Array': [13, 10, 6]}"
2861093746001578936,-#1{{~1~2 a -}}#2 de resistencia{{~p}}{{~z}} a...,Effects,246,1067169,0,56,1,-,0,1,...,-1,0,103600,-1.0,-1,0,0,34,0,"{'Array': [6, 12]}"


In [ ]:
for i in sorted(effect_df["characteristic"].unique()):
    if i == 0:
        continue
    print(i, effect_df[effect_df["characteristic"]==i]["name"].unique())

1 ['-#1{{~1~2 a -}}#2 PA' '#1{{~1~2 a }}#2 PA'
 '-#1{{~1~2 a }}#2 PA para el lanzador']
10 ['#1{{~1~2 a }}#2 fuerza' '-#1{{~1~2 a -}}#2 fuerza'
 '#1{{~1~2 a -}}#2 robo de fuerza' '+#1 de fuerza adicional']
11 ['#1{{~1~2 a }}#2 vitalidad' '-#1{{~1~2 a -}}#2 vitalidad'
 '#1{{~1~2 a -}}#2 robo de vitalidad' '+#1 vitalidad'
 '-#1{{~1~2 a -}}#2% vitalidad' '#1{{~1~2 a }}#2% vitalidad']
12 ['#1{{~1~2 a }}#2 sabiduría' '-#1{{~1~2 a -}}#2 sabiduría'
 '#1{{~1~2 a -}}#2 robo de sabiduría' '+#1 sabiduría']
13 ['#1{{~1~2 a }}#2 suerte' '-#1{{~1~2 a -}}#2 suerte'
 '#1{{~1~2 a -}}#2 robo de suerte' '+#1 suerte']
14 ['#1{{~1~2 a }}#2 agilidad' '-#1{{~1~2 a -}}#2 agilidad'
 '#1{{~1~2 a -}}#2 robo de agilidad' '+#1 de agilidad adicional']
15 ['#1{{~1~2 a }}#2 inteligencia' '-#1{{~1~2 a -}}#2 inteligencia'
 '#1{{~1~2 a -}}#2 robo de inteligencia' '+#1 inteligencia']
16 ['-#1{{~1~2 a }}#2 de daños recibidos'
 '#1{{~1~2 a }}#2 de daño{{~p}}{{~z}}' '#1{{~1~2 a }}#2 daños'
 'Daños físicos del lanzador aumen

In [ ]:
effect_df[effect_df["characteristic"].isin(range(10,16))]

,name,type,id,descriptionId,iconId,characteristic,category,characteristicOperator,showInTooltip,useDice,...,bonusType,useInFight,effectPriority,effectPowerRate,elementId,isInPercent,hideValueInTooltip,textIconReferenceId,effectTriggerDuration,actionFiltersId
2861093746001578832,#1{{~1~2 a }}#2 fuerza,Effects,118,1067135,0,10,0,+,0,1,...,1,0,5300,1.0,1,0,0,5,0,"{'Array': [7, 13]}"
2861093746001578833,#1{{~1~2 a }}#2 agilidad,Effects,119,1067139,0,14,0,+,0,1,...,1,0,5900,1.0,4,0,0,6,0,"{'Array': [4, 13]}"
2861093746001578837,#1{{~1~2 a }}#2 suerte,Effects,123,1066395,0,13,0,+,0,1,...,1,0,5700,1.0,3,0,0,3,0,"{'Array': [6, 13]}"
2861093746001578838,#1{{~1~2 a }}#2 sabiduría,Effects,124,1066476,0,12,0,+,0,1,...,1,0,6100,3.0,-1,0,0,29,0,{'Array': [13]}
2861093746001578839,#1{{~1~2 a }}#2 vitalidad,Effects,125,1066464,0,11,0,+,0,1,...,1,0,5000,0.2,-1,0,0,10,0,{'Array': [13]}
2861093746001578840,#1{{~1~2 a }}#2 inteligencia,Effects,126,1067133,0,15,0,+,0,1,...,1,0,5500,1.0,2,0,0,4,0,"{'Array': [5, 13]}"
2861093746001578862,-#1{{~1~2 a -}}#2 suerte,Effects,152,1066497,0,13,0,-,0,1,...,-1,0,5800,-0.5,3,0,0,3,0,"{'Array': [6, 12]}"
2861093746001578863,-#1{{~1~2 a -}}#2 vitalidad,Effects,153,1068521,0,11,0,-,0,1,...,-1,0,5100,-0.1,-1,0,0,10,0,{'Array': [12]}
2861093746001578864,-#1{{~1~2 a -}}#2 agilidad,Effects,154,1068112,0,14,0,-,0,1,...,-1,0,6000,-0.5,4,0,0,6,0,"{'Array': [4, 12]}"
2861093746001578865,-#1{{~1~2 a -}}#2 inteligencia,Effects,155,1068107,0,15,0,-,0,1,...,-1,0,5600,-0.5,2,0,0,4,0,"{'Array': [5, 12]}"


In [ ]:
item_df[item_df["typeId"]==18]["name"]

2861093746004721881                     Wauwau
2861093746004721896                   Miaumiau
2861093746004721913                     Wabbit
2861093746004722093              Bwak de fuego
2861093746004722094               Bwak de agua
                                ...           
2861093746004732910                    Molocán
2861093746004738370    Joven jalatín celestial
2861093746004740705           Jalatrool blanco
2861093746004741570                 Raribichón
2861093746004741592               Microsmoglob
Name: name, Length: 122, dtype: object

In [ ]:
item_df[item_df["name"].str.contains("Capa", na=False)]

,name,type,m_flags,id,nameId,typeId,descriptionId,iconId,level,realWeight,...,delay,triggers,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero
2861093746001846997,Capa de pirata,Items,24712,677.0,78774.0,17.0,47260.0,17011.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847040,Capa de Bowis,Items,24712,744.0,35685.0,17.0,35686.0,17001.0,20.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847048,Capa de vampiro,Items,24712,754.0,42701.0,17.0,42702.0,17017.0,29.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847051,Capa de justiciero,Items,24712,758.0,261306.0,17.0,966930.0,17002.0,30.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847052,Capa de pohoyo titiritasado,Items,24712,759.0,78716.0,17.0,78717.0,17003.0,33.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2861093746004741268,Capa matizada del Demonio I,Items,24584,31539.0,1129419.0,247.0,1129435.0,17645.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746004741461,Caparazón alado del Bzupervibzor,Items,24712,31750.0,1138744.0,17.0,1142458.0,17655.0,178.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746004741601,Capa de la Demonio II,Items,24584,31896.0,1144700.0,247.0,1144710.0,17643.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746004741605,Capa matizada de la Demonio II,Items,24584,31900.0,1144708.0,247.0,1144711.0,17644.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
item_df[item_df["typeId"]==17]

,name,type,m_flags,id,nameId,typeId,descriptionId,iconId,level,realWeight,...,delay,triggers,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero
2861093746001846997,Capa de pirata,Items,24712,677.0,78774.0,17.0,47260.0,17011.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847040,Capa de Bowis,Items,24712,744.0,35685.0,17.0,35686.0,17001.0,20.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847048,Capa de vampiro,Items,24712,754.0,42701.0,17.0,42702.0,17017.0,29.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847051,Capa de justiciero,Items,24712,758.0,261306.0,17.0,966930.0,17002.0,30.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001847052,Capa de pohoyo titiritasado,Items,24712,759.0,78716.0,17.0,78717.0,17003.0,33.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2861093746004738481,Capa Anteón,Items,24584,27552.0,971968.0,17.0,982345.0,17567.0,80.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746004741461,Caparazón alado del Bzupervibzor,Items,24712,31750.0,1138744.0,17.0,1142458.0,17655.0,178.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746004741480,Portal de los viajeros,Items,24712,31769.0,1138818.0,17.0,1141867.0,81075.0,197.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746004741504,Cesta de bzuevos,Items,24712,31793.0,1141009.0,17.0,1141841.0,81074.0,172.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
